# 03 - Prompt iteration: track the prompt-vs-score curve

Owner: Asad. Loads every benchmark run from `evaluation/results/runs/` and plots
how accuracy (execution / ast / partial) moves as prompts and the few-shot bank
are tuned during Day 2. Annotate each run by editing the prompt then `make benchmark`.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT/'src'))
RUNS = ROOT/'evaluation'/'results'/'runs'
runs=sorted(RUNS.glob('*.json'))
print('benchmark runs found:', len(runs))

In [ ]:
def metric(d,name):
    return next((m['value'] for m in d.get('metrics',[]) if m['name']==name), None)
series=[]
for r in runs:
    d=json.loads(r.read_text())
    series.append({'run_id':d['run_id'],'git':d.get('git_sha'),
                   'execution_accuracy':metric(d,'execution_accuracy'),
                   'ast_match':metric(d,'ast_match'),
                   'partial_match':metric(d,'partial_match'),
                   'pass_rate':d['n_passed']/d['n_questions'] if d['n_questions'] else 0,
                   'cost':metric(d,'token_cost_per_request_usd')})
for s in series: print(s)

In [ ]:
# Plot the accuracy curve across runs
try:
    import matplotlib.pyplot as plt
    if series:
        x=list(range(len(series)))
        for key in ('execution_accuracy','ast_match','partial_match'):
            ys=[s[key] or 0 for s in series]
            plt.plot(x, ys, marker='o', label=key)
        plt.xticks(x, [s['run_id'][-7:] for s in series], rotation=45)
        plt.ylim(0,1.05); plt.ylabel('score'); plt.title('Prompt-vs-score curve'); plt.legend(); plt.tight_layout(); plt.show()
    else:
        print('No runs yet - run `make benchmark` (or `python scripts/run_benchmark.py --stub`).')
except Exception as e:
    print('matplotlib not available:', e)

## Per-tier drill-down (latest run)

In [ ]:
if runs:
    d=json.loads(runs[-1].read_text())
    print('latest run', d['run_id'], 'pass_rate', round(d['n_passed']/max(d['n_questions'],1),3))
    for tier in ('easy','medium','hard'):
        v=metric(d, f'execution_accuracy_{tier}')
        if v is not None: print(f'  {tier:8} execution_accuracy = {v}')
    fails=[c['question_id'] for c in d['cases'] if not c['passed']]
    print('  failing:', fails)
else:
    print('no runs yet')